In [1]:
import numpy as np
import torch

In [4]:
def calculate_dim(input_height, input_width, input_channels, n_filter, kernel_size, stride=1, padding=0, bias = 1):
    
    if input_height + 2*padding < kernel_size or input_width + 2*padding < kernel_size:
        raise ValueError("kernel size cannot be greater than input height or width")
    
    output_height = int(((input_height - kernel_size + 2*padding)/stride) + 1)
    output_weight = int(((input_width - kernel_size + 2*padding)/stride) + 1)
    params = input_channels * n_filter * (kernel_size * kernel_size) + (n_filter * bias)
    
    return ([output_height, output_weight], params)

In [9]:
test_cases = [
    dict(input_height=9, input_width=9, input_channels=1,
         n_filter=4, kernel_size=3, stride=1, padding=1, bias=1),

    dict(input_height=8, input_width=10, input_channels=3,
         n_filter=6, kernel_size=3, stride=2, padding=0, bias=1),

    dict(input_height=12, input_width=16, input_channels=4,
         n_filter=8, kernel_size=5, stride=1, padding=0, bias=0),
]

for case in test_cases:
    print(calculate_dim(**case))

([9, 9], 40)
([3, 4], 168)
([8, 12], 800)


In [22]:
print(calculate_dim(input_height=2, input_width=4,input_channels=3, n_filter=4, kernel_size=3))

ValueError: kernel size cannot be greater than input height or width

In [18]:
print(calculate_dim(input_height=2, input_width=4,input_channels=3, n_filter=4, kernel_size=4, padding=1))

([1, 3], 196)


In [23]:
print(calculate_dim(input_height=2, input_width=4,input_channels=3, n_filter=4, kernel_size=5, padding=1))

ValueError: kernel size cannot be greater than input height or width

In [58]:
device = 'cuda'

In [ ]:
def conv2d_single_channel(kernel, x):
    
    H, W = x.shape
    Kh, Kw = kernel.shape
    
    if Kh > H or Kw > W:
        raise ValueError("Kernel cannot be greater than the image")
    
    out_h = H - Kh + 1
    out_w = W - Kw + 1
    print(x.dtype)
    output = torch.zeros((out_h, out_w), dtype=x.dtype, device=x.device)
    
    for i in range(out_h):
        for j in range(out_w):
            patch = x[i:i + Kh, j:j + Kw]
            
            output[i, j] = torch.sum(patch * kernel)
            # print(patch, kernel)
            
    return output


In [60]:
kernel = torch.tensor([[1,0],[0,-1]], device=device)
x = torch.tensor([[1,2,3],[4,5,6],[7,8,9]], device=device)

output = conv2d_single_channel(kernel, x)

print("Input Image Shape:", x.shape)
print("Kernel Shape:", kernel.shape)
print("Output Shape:", output.shape)
print("\nOutput Feature Map:\n", output)

torch.int64
Input Image Shape: torch.Size([3, 3])
Kernel Shape: torch.Size([2, 2])
Output Shape: torch.Size([2, 2])

Output Feature Map:
 tensor([[-4, -4],
        [-4, -4]], device='cuda:0')


In [61]:
kernel = torch.tensor([[1,2],[-1,0]], device=device)
x = torch.tensor([[1,2,0],[3,1,4],[2,5,1]], device=device)


output = conv2d_single_channel(kernel, x)

print("Input Image Shape:", x.shape)
print("Kernel Shape:", kernel.shape)
print("Output Shape:", output.shape)
print("\nOutput Feature Map:\n", output)

torch.int64
Input Image Shape: torch.Size([3, 3])
Kernel Shape: torch.Size([2, 2])
Output Shape: torch.Size([2, 2])

Output Feature Map:
 tensor([[2, 1],
        [3, 4]], device='cuda:0')


In [63]:
kernel = torch.tensor([[1,2,3,4],[-1,0,6,7],[-1,0,6,7],[-1,0,6,7]], device=device)
x = torch.tensor([[1,2,0],[3,1,4],[2,5,1]], device=device)


output = conv2d_single_channel(kernel, x)

print("Input Image Shape:", x.shape)
print("Kernel Shape:", kernel.shape)
print("Output Shape:", output.shape)
print("\nOutput Feature Map:\n", output)

ValueError: Kernel cannot be greater than the image

In [67]:
def conv2d_multi_channel(kernel, x):
    
    C_in, H, W = x.shape
    C_out, Ch, Kh, Kw = kernel.shape
    
    if C_in != Ch:
        raise ValueError(f"Input channel mismatch: image has {C_in}, kernel has {Ch}")
    if Kh > H or Kw > W:
        raise ValueError("Kernel cannot be greater than the image")
    
    out_h = H - Kh + 1
    out_w = W - Kw + 1
    
    output = torch.zeros((C_out, out_h, out_w), dtype=x.dtype, device=x.device)
    
    for k in range(C_out):
        for i in range(out_h):
            for j in range(out_w):
                patch = x[:, i:i + Kh, j:j + Kw]
                
                output[k, i, j] = torch.sum(patch * kernel[k])
                # print(patch, kernel)
            
    return output

In [ ]:
x = torch.ones((3, 5, 5))

kernel_1 = torch.stack([
    torch.ones((3, 3)),
    torch.full((3, 3), 2.0),
    torch.full((3, 3), 3.0),
]).unsqueeze(0)

kernel_2 = torch.stack([
    torch.zeros((3, 3)),
    torch.zeros((3, 3), 2.0),
    torch.zeros((3, 3), 3.0),
]).unsqueeze(0)

kernel = torch.cat((kernel_1,kernel_2), dim=0)

print(x.shape)       # (3, 5, 5)
print(kernel.shape)  # (1, 3, 3, 3)

output = conv2d_multi_channel(kernel, x)
print(output.shape)
print(output)

torch.Size([3, 5, 5])
torch.Size([2, 3, 3, 3])
torch.Size([2, 3, 3])
tensor([[[54., 54., 54.],
         [54., 54., 54.],
         [54., 54., 54.]],

        [[45., 45., 45.],
         [45., 45., 45.],
         [45., 45., 45.]]])


In [71]:
x = torch.ones((3, 5, 5))

kernel_1 = torch.stack([
    torch.ones((3, 3)),
    torch.full((3, 3), 2.0),
    torch.full((3, 3), 3.0),
]).unsqueeze(0)

kernel_2 = torch.stack([
    torch.zeros((3, 3)),
    torch.zeros((3, 3)),
    torch.zeros((3, 3)),
]).unsqueeze(0)

kernel = torch.cat((kernel_1,kernel_2), dim=0)

print(x.shape)       # (3, 5, 5)
print(kernel.shape)  # (1, 3, 3, 3)

output = conv2d_multi_channel(kernel, x)
print(output.shape)
print(output)

torch.Size([3, 5, 5])
torch.Size([2, 3, 3, 3])
torch.Size([2, 3, 3])
tensor([[[54., 54., 54.],
         [54., 54., 54.],
         [54., 54., 54.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.]]])
